## Basic scripts

This contains the basic scripts to be used regardless of the metric to be applied

In [ ]:
import os
import pandas as pd 
import numpy as np  
#import text2term

import ast
import csv
import json

from collections import Counter, deque

### Import and clean up the data

In [ ]:
## Clean up the results from the curation of GPT prediction

def clean_curation_results(topic_list):
    if topic_list == 'None':
        clean_list = []
    else:
        temp_list = topic_list.replace('[','').replace(']','')
        dirty_list = temp_list.split('|')
        clean_list = [x.strip().strip('{').strip('}') for x in dirty_list]
    clean_set = set(clean_list)
    return clean_set

In [ ]:
## All files to be evaluated have the primary curator's classifications in the sheet labeld 'g_rating'
## If the classification was performed by the secondary curator, the sheetname is 'j_rating'
## For classifications performed by GPT, the sheetname is 'predictions'

def import_classifications(filename,sheetname):
    script_path = os.getcwd()
    data_path = os.path.join(script_path,'data')
    j_ratings = pd.read_excel(os.path.join(data_path, filename), sheetname, engine='openpyxl')
    j_ratings.fillna('None',inplace=True)
    j_ratings['Curator_1'] = j_ratings.apply(lambda row: clean_curation_results(row['Topics']), axis=1)
    curator1 = j_ratings[['Data Repository','Name','Description','Curator_1']].copy()
    g_ratings = pd.read_excel(os.path.join(data_path,'Curator classifications.xlsx'), 'g_rating', engine='openpyxl')
    g_ratings.fillna('None',inplace=True)
    g_ratings['Curator_2'] = g_ratings.apply(lambda row: clean_curation_results(row['Topics']), axis=1)
    curator2 = g_ratings[['Data Repository','Name','Description','Curator_2']].copy()
    data_df = curator1.merge(curator2,on=['Data Repository','Name','Description'],how='inner')
    data_df['Evaluation_type'] = filename
    return data_df

## Basic metrics
Precision, Recall, Jaccard Similarity


### Calculate Jaccard similarity

In [1]:
def jaccard_similarity(row):
    set1, set2 = row['Curator_1'], row['Curator_2']
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2)) 
    return intersection / union

## only used for alternative weighted similarity calculation (alt_sim)
def count_matches(row):
    set1, set2 = row['Curator_1'], row['Curator_2']
    intersection = len(set1.intersection(set2))    
    return intersection

def count_terms(row):
    set1, set2 = row['Curator_1'], row['Curator_2']
    union = len(set1.union(set2))
    return union

In [ ]:
data_df['Jaccard Similarity'] = data_df.apply(jaccard_similarity, axis=1)
## only used for alternative weighted similarity calculation (alt sim)
data_df['match_count'] = data_df.apply(count_matches, axis=1)
data_df['term_count'] = data_df.apply(count_terms, axis=1)

### Precision, recall, F

In [ ]:
def calculate_precision_recall_per_row(ground_truth, predictions):
    precision_per_row = []
    recall_per_row = []

    for truth_labels, predicted_labels in zip(ground_truth, predictions):

        true_positives = len(truth_labels.intersection(predicted_labels))
        false_positives = len(predicted_labels - truth_labels)
        false_negatives = len(truth_labels - predicted_labels)

        precision = true_positives / (true_positives + false_positives) if (true_positives + false_positives) != 0 else 0
        recall = true_positives / (true_positives + false_negatives) if (true_positives + false_negatives) != 0 else 0

        precision_per_row.append(precision)
        recall_per_row.append(recall)

    return precision_per_row, recall_per_row


precision, recall = calculate_precision_recall_per_row(data_df['Curator_1'],data_df['Curator_2'])

#print(f'Precision: {precision}')
#print(f'Recall: {recall}')
data_df['Precision'] = precision
data_df['Recall'] = recall

## Advanced metrics for evaluating conceptual similarity

1. Identify the exact matches between the two classifications (eg between predicted and curated or between curator A and curator B)
2. Check if the predicted (or curator A) term is within the same tree as the primary curator's term
3. If two terms are within the same tree:
    * Treat the two terms as a match
    * Determine which term is closer to the root (http://edamontology.org/topic_0003)
        - Calculate a weight to apply depending on how close the closest term is to the root (Weight 1)
          - If closest term is one step away from root it should have a lower score than a closest term that is 2 steps away. This is to lower the weight of excessively generic terms (use: 1-(1/(# of steps to closest term))
        - Calculate a weight to apply depending on the number of steps between the two 'matching' terms (Weight 2)
            - If the closest term to root is the ground truth term, use: (1/(# of steps between the terms))
            - If the closest term to root is the prediction term, use: -(1/(# of steps between the terms)): It is negative only to ensure we will later be able to inspect the directionality
        - The overall weight should be a combination of the two, for example:
            - Overall weight = Weight 1 + ABS(Weight 2)

### Calculating the conceptual similarity scores

#### Calculating the simple weighted similarity score
This is a straightforward metric with a strong bias for exact matches
1. After removing the exact matches, perform pairwise matches, calculating the weight for each match
2. simple weighted similarity = Add the overall weight for all matches and divide by the total number of matches, then add the j-sim (since we previously removed exact matches)
    * This method essentially averages the weighted similarity across all pairwise matches and *then* adds back the J-sim values since exact matches were removed). This approach thus gives a much higher score for records with exact matches

#### Calculating the conceptual similarity score
This similarity score includes the exact matches prior to averaging out the results of all the pairwise matches 
1. After removing the exact matches, perform pairwise matches, calculating the weight for each match
2. Conceptual similarity score = Add the overall weight for all the pairwise matches to the number of exact matches *then* divide by the sum of the total number of pairwise matches and exact matches

#### Calculating the adjusted similarity score
This similarity score includes the exact matches in the pairwise combinations potentially diluting the value of an exact match
1. Adjusted similarity score = Add the overall weight for all pairwise matches (without removing exact matches) *then* divide by the total number of pairwise matches




### Additional adjustments and evaluations

**Adjusting for prediction number biases:**

- The weighted similarity will be advantageous to ChatGPT 4 due to its tendency to dump every relevant term
- To account for that, we can penalize it for making excessive guesses by multiplying the weighted similarity against the ratio of (# of gold standard terms)/max(# of gold standard terms),(# of predicted terms)).
  * If LLM guesses the same number of terms as # of gold standard, this ratio = 1: No penalty
  * If LLM guesses more terms than # of gold standard, this ratio >1: penalty, greater number, greater penalty
  
**Evaluating whether the prediction has a tendency to be more specific or less specific than the gold standard**

- Broadness evaluation: (# of positive Weight 2 values)/(# of negative Weight 2 values)
  - Number of times broader term is ground truth / Number of times broader term is predicted term
- If broadness evaluation is \>1, LLM model predictions are more specific than Ground truth/gold standard terms
  - If broadness evaluation is \<1, LLM model predictions are less specific than ground truth/gold standard terms

1. Calculate J-similarity, precision, recall and then remove the terms in common

2. Check if a prediction term is within the same tree as a ground truth term

3. If two terms are within the same tree:

- Treat the two terms as a match
- Determine which term is closer to the root ([http://edamontology.org/topic\_0003](http://edamontology.org/topic_0003))
  - Calculate a weight to apply depending on how close the closest term is to the root (Weight 1)
    - If closest term is one step away from root it should have a lower score than a closest term that is 2 steps away. This is to lower the weight of excessively generic terms (use: 1-(1/(# of steps to closest term))
  - Calculate a weight to apply depending on the number of steps between the two 'matching' terms (Weight 2)
    - If the closest term to root is the ground truth term, use: (1/(# of steps between the terms))
    - If the closest term to root is the prediction term, use: -(1/(# of steps between the terms)): It is negative only to ensure we will later be able to inspect the directionality
- The overall weight should be a combination of the two, for example:
- Overall weight = Weight 1 + ABS(Weight 2)
- Weighted similarity: Add the overall weight for all matches and divide by the total number of matches, then add the j-sim (since we previously removed exact matches)
  * Note, investigated this as an alternative, but it doesn't account for the fact that when you do pairwise matches, it's possible that terms will match more than once.:
    - alt sim: (Overall weight)/(len(union(terms))) + J-sim, or (overall weight + match count)/len(union(terms))) 

**Adjusting for prediction number biases:**

- The weighted similarity will be advantageous to ChatGPT 4 due to its tendency to dump every relevant term
- To account for that, we can penalize it for making excessive guesses by multiplying the weighted similarity against the ratio of (# of gold standard terms)/max(# of gold standard terms),(# of predicted terms)).
  * If LLM guesses the same number of terms as # of gold standard, this ratio = 1: No penalty
  * If LLM guesses more terms than # of gold standard, this ratio >1: penalty, greater number, greater penalty

**Evaluating whether the prediction has a tendency to be more specific or less specific than the gold standard**

- Broadness evaluation: (# of positive Weight 2 values)/(# of negative Weight 2 values)
  - Number of times broader term is ground truth / Number of times broader term is predicted term
- If broadness evaluation is \>1, LLM model predictions are more specific than Ground truth/gold standard terms
  - If broadness evaluation is \<1, LLM model predictions are less specific than ground truth/gold standard terms

### Calculating the Weights

In [ ]:
# Calculate weights for each row
def calculate_weights(tree, term1, term2, paths=shortest_paths):
    # Weight 1: Distance from root
    steps_to_term1 = shortest_distance(tree, term1)
    steps_to_term2 = shortest_distance(tree, term2)
    # determine the steps between the two terms
    def steps_between_terms(paths, topic1, topic2):
        path1 = paths[topic1]
        path2 = paths[topic2]
        index1 = index2 = 0
        for i, (n1, n2) in enumerate(zip(path1, path2)):
            if n1 != n2:
                break
            index1 = i + 1
            index2 = i + 1
        return len(path1) - index1 + len(path2) - index2
    steps_between = steps_between_terms(paths, term1, term2)
    # Assuming term1 is ground truth and term2 is prediction
    if steps_to_term1 < steps_to_term2:
        weight1 = 1 - (1 / steps_to_term1)
        weight2 = (1 / steps_between)
    else:
        weight1 = 1 - (1 / steps_to_term2)
        weight2 = -1 / steps_between   
    weight = weight1 + abs(weight2)
    return weight, (weight1, weight2)

### Identifying the classifications/predictions unique vs in common to each curator/GPT

In [ ]:
def process_terms(data_df):
    reviewer1 = data_df['Curator_1']
    reviewer2 = data_df['Curator_2']

    exclusive_reviewer1 = []
    exclusive_reviewer2 = []

    for reviewer1, reviewer2 in zip(reviewer1, reviewer2):
        exclusive_reviewer1.append(set([label for label in reviewer1 if label not in reviewer2]))
        exclusive_reviewer2.append(set([label for label in reviewer2 if label not in reviewer1]))
    data_df['Exclusive Curator 1'] = exclusive_reviewer1
    data_df['Exclusive Curator 2'] = exclusive_reviewer2
    return data_df

### Calculating the simple weighted similarity score

In [ ]:
def calculating_metrics(data_df, metric_type):
    data_df['Jaccard Similarity'] = data_df.apply(jaccard_similarity, axis=1)
    data_df['Metric_used'] = "metric_type"
    for idx, row in data_df.iterrows():
        if metric_type == "simple_weighted_similarity":
            ground_truth = row['Exclusive Curator 1']
            prediction = row['Exclusive Curator 2']
        elif metric_type == "conceptual_similarity":
            ground_truth = row['Exclusive Curator 1']
            prediction = row['Exclusive Curator 2']
            data_df['match_count'] = data_df.apply(count_matches, axis=1)
            data_df['term_count'] = data_df.apply(count_terms, axis=1)
        elif metric_type == "adjusted_weight_similarity":
            ground_truth = row['Curator 1']
            prediction = row['Curator 2'] 

        weights = []
        num_positive_w2 = num_negative_w2 = 0
        for truth_label in ground_truth:
            for pred_label in prediction:
                truth_topic, pred_topic = plabel_topic_dict[truth_label], plabel_topic_dict[pred_label]
                # If labels are not in the same subtree
                if not set(topic_subtree_dict[pred_topic]) & set(topic_subtree_dict[truth_topic]):
                    continue
                total_weight, (w1, w2) = calculate_weights(topic_dict, truth_topic, pred_topic)
                weights.append(total_weight)

                if w2 >= 0:
                    num_positive_w2 += 1
                elif w2 < 0:
                    num_negative_w2 += 1
        if metric_type == "simple weighted similarity":
            data_df.loc[idx, 'Base_score'] = sum(weights)/max(1,(len(weights))) + row['Jaccard Similarity'] ## use 1 if no weights
        elif metric_type == "conceptual similarity":
            data_df.loc[idx, 'Base_score'] = (sum(weights)+row['match_count'])/((len(weights))+row['match_count'])
        elif metric_type == "adjusted weight similarity":
            data_df.loc[idx, 'Base_score'] = sum(weights)/(len(weights))
        try:
            data_df.loc[idx, 'Broadness Score'] = num_positive_w2 / (num_negative_w2)
        except:
            data_df.loc[idx, 'Broadness Score'] = 0

        # Adjust weights for prediction number bias, penalized overprediction without incentivizing underprediction
        data_df['overprediction penalty'] = data_df['Curator_1'].apply(len) / data_df['Curator_2'].apply(len)
        data_df['penalty'] = data_df['overprediction penalty'].apply(lambda x: 1 if x>1 else x)
        data_df['Adjusted_score'] = data_df['Base_score'] * data_df['penalty']
        data_df.to_csv(os.path.join('result',f'{data_df.iloc[0]['Evaluation_type']}_{metric_type}_results.tsv'), sep='\t', header=True, index=False)
        return data_df